# Profiling and benchmarking hygiene on a T4

Companion to the card **profiling.html** (perf-3-kernels).

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

What you will measure:

1. A naive and a shared-memory transpose (4096 × 4096 floats), timed with `cudaEvent` after warm-up, GB/s next to the T4's 320 GB/s.
2. The same program under **Nsight Systems** (`nsys profile --stats=true`): where did the time go?
3. One kernel under **Nsight Compute** (`ncu --section SpeedOfLight --section MemoryWorkloadAnalysis`): what bounds it?
4. **Compute Sanitizer** on four deliberately broken kernels: memcheck, racecheck, initcheck, synccheck.
5. In PyTorch: `time.time()` around an async launch vs CUDA events vs `triton.testing.do_bench`, and a hot-L2 vs cold-L2 GEMV.

Colab images may not include `nsys` or `ncu`, and a VM may refuse `ncu` access to the performance counters (`ERR_NVGPUCTRPERM`). The cells below check first and skip with a message rather than fail. This notebook has not been run by the author; your run is the exercise.

In [ ]:
!nvidia-smi

In [ ]:
# Which tools does this image have?
import shutil, subprocess, os
os.environ["PATH"] += ":/usr/local/cuda/bin"
for t in ["nvcc", "nsys", "ncu", "compute-sanitizer"]:
    print(f"{t:18s}", shutil.which(t) or "NOT FOUND")
print(subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout)

If `nsys` or `ncu` is missing, the next cell tries NVIDIA's apt packages that match the installed CUDA version (`cuda-nsight-systems-X-Y`, `cuda-nsight-compute-X-Y`). If the install fails, skip sections 2 and 3; everything else still runs.

In [ ]:
import re, subprocess, shutil
TRY_INSTALL = True
if TRY_INSTALL and not (shutil.which("nsys") and shutil.which("ncu")):
    v = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
    m = re.search(r"release (\d+)\.(\d+)", v)
    if m:
        pk = f"{m.group(1)}-{m.group(2)}"
        !apt-get -qq update > /dev/null
        !apt-get -qq install -y cuda-nsight-systems-{pk} cuda-nsight-compute-{pk} > /dev/null || echo "install failed: skip sections 2 and 3"
        !ls -d /opt/nvidia/nsight-systems/*/bin /opt/nvidia/nsight-compute/* 2>/dev/null
import glob, os
for d in glob.glob("/opt/nvidia/nsight-systems/*/bin") + glob.glob("/opt/nvidia/nsight-compute/*"):
    os.environ["PATH"] += ":" + d
for t in ["nsys", "ncu"]:
    print(f"{t:6s}", shutil.which(t) or "still NOT FOUND (skip that section)")

## 1. Two transposes, timed properly

Warm-up launches first, then a timed loop between two `cudaEvent`s, every CUDA call checked, the result checked against the CPU. Pass `once` to launch each kernel exactly once (handy under ncu).

In [ ]:
%%writefile transpose.cu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>
#include <cuda_runtime.h>

#define CUDA_CHECK(e) do { cudaError_t r_ = (e); if (r_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(r_)); \
  exit(1); } } while (0)

const int N = 4096, TILE = 32, ROWS = 8;

// reads rows (coalesced), writes columns (strided): one 32-byte sector per float written
__global__ void transposeNaive(float *out, const float *in) {
  int x = blockIdx.x * TILE + threadIdx.x;
  int y = blockIdx.y * TILE + threadIdx.y;
  for (int j = 0; j < TILE; j += ROWS)
    out[x * N + (y + j)] = in[(y + j) * N + x];
}

// stages a 32x32 tile in shared memory (+1 column of padding against bank conflicts)
__global__ void transposeSmem(float *out, const float *in) {
  __shared__ float tile[TILE][TILE + 1];
  int x = blockIdx.x * TILE + threadIdx.x;
  int y = blockIdx.y * TILE + threadIdx.y;
  for (int j = 0; j < TILE; j += ROWS)
    tile[threadIdx.y + j][threadIdx.x] = in[(y + j) * N + x];
  __syncthreads();
  x = blockIdx.y * TILE + threadIdx.x;
  y = blockIdx.x * TILE + threadIdx.y;
  for (int j = 0; j < TILE; j += ROWS)
    out[(y + j) * N + x] = tile[threadIdx.x][threadIdx.y + j];
}

typedef void (*Kern)(float *, const float *);

float timeKernel(Kern k, float *d_out, const float *d_in, int warm, int reps) {
  dim3 grid(N / TILE, N / TILE), block(TILE, ROWS);
  for (int i = 0; i < warm; i++) k<<<grid, block>>>(d_out, d_in);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());
  cudaEvent_t a, b;
  CUDA_CHECK(cudaEventCreate(&a)); CUDA_CHECK(cudaEventCreate(&b));
  CUDA_CHECK(cudaEventRecord(a));
  for (int i = 0; i < reps; i++) k<<<grid, block>>>(d_out, d_in);
  CUDA_CHECK(cudaEventRecord(b));
  CUDA_CHECK(cudaEventSynchronize(b));
  CUDA_CHECK(cudaGetLastError());
  float ms = 0; CUDA_CHECK(cudaEventElapsedTime(&ms, a, b));
  CUDA_CHECK(cudaEventDestroy(a)); CUDA_CHECK(cudaEventDestroy(b));
  return reps > 0 ? ms / reps : 0.f;
}

bool check(const std::vector<float> &h_in, float *d_out) {
  std::vector<float> h_out((size_t)N * N);
  CUDA_CHECK(cudaMemcpy(h_out.data(), d_out, h_out.size() * sizeof(float), cudaMemcpyDeviceToHost));
  for (int r = 0; r < N; r++)
    for (int c = 0; c < N; c++)
      if (h_out[(size_t)c * N + r] != h_in[(size_t)r * N + c]) return false;
  return true;
}

int main(int argc, char **argv) {
  bool once = argc > 1 && strcmp(argv[1], "once") == 0;
  size_t bytes = (size_t)N * N * sizeof(float);
  std::vector<float> h_in((size_t)N * N);
  for (size_t i = 0; i < h_in.size(); i++) h_in[i] = (float)(i % 1000003);
  float *d_in, *d_out;
  CUDA_CHECK(cudaMalloc(&d_in, bytes)); CUDA_CHECK(cudaMalloc(&d_out, bytes));
  CUDA_CHECK(cudaMemcpy(d_in, h_in.data(), bytes, cudaMemcpyHostToDevice));
  int warm = once ? 1 : 10, reps = once ? 0 : 100;
  const char *names[2] = {"naive", "smem "};
  Kern ks[2] = {transposeNaive, transposeSmem};
  for (int k = 0; k < 2; k++) {
    CUDA_CHECK(cudaMemset(d_out, 0, bytes));
    float ms = timeKernel(ks[k], d_out, d_in, warm, reps);
    bool ok = check(h_in, d_out);
    if (once) { printf("%s ran once, correct=%s\n", names[k], ok ? "yes" : "NO"); continue; }
    double gbs = 2.0 * bytes / (ms * 1e-3) / 1e9;   // read N*N floats + write N*N floats
    printf("%s  %.3f ms  %.1f GB/s  = %.0f%% of 320 GB/s   correct=%s\n",
           names[k], ms, gbs, 100.0 * gbs / 320.0, ok ? "yes" : "NO");
  }
  CUDA_CHECK(cudaFree(d_in)); CUDA_CHECK(cudaFree(d_out));
  return 0;
}

In [ ]:
!nvcc -O3 -lineinfo -arch=sm_75 -o transpose transpose.cu && ./transpose

## 2. Nsight Systems: where did the time go?

`nsys` records one pass of the real program: CPU calls, kernel launches, memcpys and the gaps between them. It does not lock clocks, flush caches or serialize anything. `--stats=true` prints summary tables (CUDA API calls, GPU kernels, memory operations) after the run; download `transpose_nsys.nsys-rep` and open it in the Nsight Systems GUI to see the timeline.

In [ ]:
import shutil
if shutil.which("nsys"):
    !nsys profile --stats=true -o transpose_nsys --force-overwrite true ./transpose
else:
    print("nsys not available on this image: skipped")

Look for: the two kernels' total time, the `cudaMemcpy` host-to-device and device-to-host rows (the 64 MB copies plus the check), and how much of the wall-clock is CPU work (the CPU fills and checks 16M floats, which is slow). In a real app, this table is how you pick the kernel to study next.

## 3. Nsight Compute: one kernel under a microscope

`ncu` replays each kernel (saving and restoring its memory between passes), flushes caches before each pass (`--cache-control all`), serializes launches and locks clocks (default `base` through 2025.4, `boost` from 2026.1). We run `./transpose once` so each kernel launches only a few times. `--list-sections` prints the section identifiers your version knows.

In [ ]:
import shutil
if shutil.which("ncu"):
    !ncu --version
    !ncu --list-sections | head -40
    !ncu --section SpeedOfLight --section MemoryWorkloadAnalysis -k regex:transpose -c 2 -o transpose_ncu -f ./transpose once
    !ncu --import transpose_ncu.ncu-rep --page details | head -120
else:
    print("ncu not available on this image: skipped")

If you see `ERR_NVGPUCTRPERM`, the VM does not let user code read the GPU performance counters; there is nothing to fix from inside Colab, so skip this section (the H100 script on the card does this part on a machine you control).

Compare the two kernels' **Memory Throughput %** and **DRAM Throughput %** in Speed of Light, and the sectors-per-request numbers in Memory Workload Analysis: the naive kernel's strided writes waste most of each 32-byte sector. Also compare ncu's **Duration** with the event timing from section 1: they differ because of the clock lock and the cache flush.

## 4. Compute Sanitizer: correctness before speed

Four tiny bugs, one per tool. Each runs once plainly (the bug is often **silent**) and once under the sanitizer.

In [ ]:
%%writefile bugs.cu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cuda_runtime.h>

#define CUDA_CHECK(e) do { cudaError_t r_ = (e); if (r_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(r_)); \
  exit(1); } } while (0)

// memcheck: off-by-one, thread n writes one float past the allocation
__global__ void oob(float *a, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i <= n) a[i] = 1.0f;          // should be i < n
}

// racecheck: block sum in shared memory with the __syncthreads() missing
__global__ void race(float *out) {
  __shared__ float s[256];
  s[threadIdx.x] = 1.0f;
  // missing __syncthreads(): thread 0 may read before others have written
  if (threadIdx.x == 0) {
    float t = 0;
    for (int k = 0; k < 256; k++) t += s[k];
    *out = t;
  }
}

// initcheck: reads device memory nobody wrote (no memcpy, no memset)
__global__ void uninit(const float *a, float *out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) out[i] = a[i] * 2.0f;
}

// synccheck: __syncthreads() inside a branch only some threads take
__global__ void divergentBarrier(float *out) {
  if (threadIdx.x < 16) {
    __syncthreads();
    out[threadIdx.x] = 1.0f;
  }
}

int main(int argc, char **argv) {
  const char *mode = argc > 1 ? argv[1] : "oob";
  const int n = 1000;
  float *a, *b;
  CUDA_CHECK(cudaMalloc(&a, n * sizeof(float)));
  CUDA_CHECK(cudaMalloc(&b, n * sizeof(float)));
  if (!strcmp(mode, "oob"))       oob<<<(n + 255) / 256, 256>>>(a, n);
  else if (!strcmp(mode, "race")) race<<<1, 256>>>(b);
  else if (!strcmp(mode, "init")) uninit<<<(n + 255) / 256, 256>>>(a, b, n);
  else if (!strcmp(mode, "sync")) divergentBarrier<<<1, 64>>>(b);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());
  printf("%s: finished with no CUDA error reported\n", mode);
  CUDA_CHECK(cudaFree(a)); CUDA_CHECK(cudaFree(b));
  return 0;
}

In [ ]:
!nvcc -O3 -lineinfo -arch=sm_75 -o bugs bugs.cu
print("--- plain runs (bugs are usually silent) ---")
!./bugs oob; ./bugs race; ./bugs init; ./bugs sync

In [ ]:
print("=== memcheck (the default tool): out-of-bounds write ===")
!compute-sanitizer ./bugs oob 2>&1 | head -25
print("=== racecheck: shared-memory hazard ===")
!compute-sanitizer --tool racecheck ./bugs race 2>&1 | head -25
print("=== initcheck: uninitialized global read ===")
!compute-sanitizer --tool initcheck ./bugs init 2>&1 | head -25
print("=== synccheck: barrier in divergent code ===")
!compute-sanitizer --tool synccheck ./bugs sync 2>&1 | head -25

Expected: memcheck reports an invalid global write of size 4 by thread 232 of block 3 (index 1000); racecheck reports a read-after-write hazard on `s`; initcheck reports uninitialized reads of `a`; synccheck reports a divergent barrier (how synccheck treats this pattern depends on the architecture and version: read what it prints). The sanitizer slows kernels a lot, so never time under it.

## 5. Timing from Python: the async pitfall, events, do_bench, hot vs cold L2

In [ ]:
import time, statistics, torch, triton
from triton.testing import do_bench

dev = torch.cuda.get_device_properties(0)
L2 = dev.L2_cache_size
print(f"{dev.name}: {dev.multi_processor_count} SMs, L2 = {L2/2**20:.1f} MiB")

a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
b = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
flops = 2 * 4096**3
for _ in range(10): a @ b
torch.cuda.synchronize()

# (a) wrong: time.time() around an asynchronous launch measures the launch only
t0 = time.time(); c = a @ b; t1 = time.time()
torch.cuda.synchronize()
print(f"time.time(), no sync      : {(t1-t0)*1e3:8.3f} ms   <- the launch, not the matmul")

# (b) better: synchronize before stopping the CPU clock
torch.cuda.synchronize(); t0 = time.time(); c = a @ b; torch.cuda.synchronize(); t1 = time.time()
print(f"time.time() with sync     : {(t1-t0)*1e3:8.3f} ms   <- includes launch + sync overhead")

# (c) CUDA events, many runs, median and p90
times = []
for _ in range(50):
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    s.record(); c = a @ b; e.record(); torch.cuda.synchronize()
    times.append(s.elapsed_time(e))        # milliseconds
times.sort()
med, p90 = statistics.median(times), times[int(0.9 * len(times)) - 1]
print(f"cuda events  median / p90 : {med:8.3f} / {p90:.3f} ms  -> {flops/med/1e9:.1f} TFLOPS vs 65 FP16 tensor peak")

# (d) do_bench: warmup=25 ms, rep=100 ms, flushes L2 before every run
q50, q90 = do_bench(lambda: a @ b, quantiles=[0.5, 0.9])
print(f"do_bench     median / p90 : {q50:8.3f} / {q90:.3f} ms  -> {flops/q50/1e9:.1f} TFLOPS")

In [ ]:
# Hot vs cold L2 on a GEMV: y = W x with W sized to about half of L2 (fits), then 16x L2 (does not)
def gemv_case(weight_bytes):
    rows = 4096
    cols = max(256, int(weight_bytes // (rows * 2)))
    W = torch.randn(rows, cols, device="cuda", dtype=torch.float16)
    x = torch.randn(cols, device="cuda", dtype=torch.float16)
    fn = lambda: W @ x
    for _ in range(20): fn()
    torch.cuda.synchronize()
    # hot: back-to-back runs, W stays in L2 between them
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    reps = 200
    s.record()
    for _ in range(reps): fn()
    e.record(); torch.cuda.synchronize()
    hot = s.elapsed_time(e) / reps
    # cold: do_bench zeroes a 256 MB buffer before each run
    cold = do_bench(fn, return_mode="median")
    wb = W.numel() * 2
    print(f"W = {wb/2**20:6.1f} MiB ({wb/L2:4.1f}x L2): hot {hot*1e3:7.1f} us ({wb/hot/1e6:6.1f} GB/s)  "
          f"cold {cold*1e3:7.1f} us ({wb/cold/1e6:6.1f} GB/s)  hot is {cold/hot:.2f}x faster  [T4 DRAM peak 320 GB/s]")

gemv_case(L2 // 2)
gemv_case(16 * L2)

Read the two lines: when W fits in L2, the back-to-back ("hot") loop reports a bandwidth that can exceed what DRAM can deliver, which is the benchmark lying. When W is 16× L2, hot and cold agree. Small kernels also carry launch overhead in both numbers, so very small W values mostly measure the launch.

## Reference numbers and try this

The card's numbers come from sources, not from this notebook: CUTLASS's methodology (rotate buffers to ≥ 2× L2, >1000 iterations at 4096³, clocks oscillate ~3 s), `triton.testing.do_bench` source (warmup=25 ms, rep=100 ms, 256 MB L2 flush, mean by default), Nsight Compute docs (replay, `--cache-control all`, clock control), and the Scaling Book's measured ~5.5 TB/s H100 L2 against 3.35 TB/s HBM (≈ 1.6×, an H100 figure, not a T4 one).

Try this:
- Re-run section 3 with `--cache-control none --clock-control none` and compare ncu's Duration with the event timing.
- In section 5, set `return_mode="min"` in do_bench and see how much rosier the min is than the median.
- Remove the `+ 1` padding in `transposeSmem` (`tile[TILE][TILE]`) and look for bank conflicts in ncu's Memory Workload Analysis.